In [146]:
import pandas as pd 
import sklearn
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 
from IPython.display import Image 
%matplotlib inline

In [147]:
import pandas as pd
data = pd.read_csv('housing_data_08-02-2021.csv', index_col=0)
data.head()

,location,district,rooms,size,level,max_levels,price
0,"город Ташкент, Юнусабадский район, Юнусабад 8-...",Юнусабадский,3,57,4,4,52000
1,"город Ташкент, Яккасарайский район, 1-й тупик ...",Яккасарайский,2,52,4,5,56000
2,"город Ташкент, Чиланзарский район, Чиланзар 2-...",Чиланзарский,2,42,4,4,37000
3,"город Ташкент, Чиланзарский район, Чиланзар 9-...",Чиланзарский,3,65,1,4,49500
4,"город Ташкент, Чиланзарский район, площадь Актепа",Чиланзарский,3,70,3,5,55000


In [148]:
def clean_data(df):
    data.drop('location',axis=1,inplace=True)
    df['price'] = df['price'].str.replace('Договорная', '0')
    df['price'] = df['price'].fillna(method='bfill')
    df['price'] = df['price'].astype(int)
    return df
data = clean_data(data)

In [149]:
from sklearn.model_selection import train_test_split    
train_set, test_set = train_test_split(data, test_size=0.2, random_state=30)

X = train_set.drop('price', axis=1)
y = train_set['price'].copy()
X_num = X.drop('district', axis=1)

In [150]:
X_num

,rooms,size,level,max_levels
6038,2,60,2,9
2058,3,80,2,4
7114,2,63,4,4
5035,2,55,4,4
6864,1,30,2,4
...,...,...,...,...
500,3,74,5,9
3885,2,40,4,4
7085,4,88,2,9
4517,3,78,7,9


In [151]:
from sklearn.base import BaseEstimator, TransformerMixin

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, clean_data):
        self.clean_data = clean_data
    def fit(self,df, y=None):
        return self
    def transform(self, df):
        df['size'] = df['size'].str.replace('Площадьземли:1сот', '0').astype(float)
        return df

In [152]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler #yoki min_max_scalerdan ham foydalansa buladi

num_pipeline = Pipeline([# 1-qadam
    ('AttributesAdder', CombinedAttributesAdder(X)),
    ('min_max_bir_deopazonga_tushurish', StandardScaler())
])
num_pipeline.fit_transform(X_num)
num_pipeline

Pipeline(steps=[('AttributesAdder',
                 CombinedAttributesAdder(clean_data=             district  rooms size  level  max_levels
6038     Юнусабадский      2   60      2           9
2058      Яшнободский      3   80      2           4
7114     Чиланзарский      2   63      4           4
5035      Мирабадский      2   55      4           4
6864     Чиланзарский      1   30      2           4
...               ...    ...  ...    ...         ...
500      Чиланзарский      3   74      5           9
3885     Чиланзарский      2   40      4           4
7085  Шайхантахурский      4   88      2           9
4517     Чиланзарский      3   78      7           9
5925     Юнусабадский      1   31      2           4

[6052 rows x 5 columns])),
                ('min_max_bir_deopazonga_tushurish', StandardScaler())])

In [153]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
num_atributs = list(X_num)
str_atributs = ['district']

full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_atributs),# 1-nomi 2-pipelineni choqiramiz 3-qaysi ustunlar ustida pipeline ishlaganini aytypapmiz
    ('categorical', OneHotEncoder(), str_atributs),
])

In [154]:
x_prepared = full_pipeline.fit_transform(X)

In [155]:
x_prepared

array([[-0.58157372, -0.03721175, -0.76158055, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.33980956, -0.02519412, -0.76158055, ...,  0.        ,
         0.        ,  1.        ],
       [-0.58157372, -0.03540911,  0.13312691, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 1.26119284, -0.02038707, -0.76158055, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.33980956, -0.02639588,  1.47518811, ...,  0.        ,
         0.        ,  0.        ],
       [-1.50295699, -0.05463732, -0.76158055, ...,  0.        ,
         0.        ,  0.        ]])

In [156]:
from sklearn.linear_model import LinearRegression 

LR_model = LinearRegression()

In [157]:
LR_model.fit(x_prepared, y)

LinearRegression()

In [158]:
test_set

,district,rooms,size,level,max_levels,price
6214,Чиланзарский,2,60,4,9,34500
3311,Мирзо-Улугбекский,2,50,1,4,29500
3066,Мирзо-Улугбекский,3,152,5,8,285000
1896,Юнусабадский,3,78,4,4,55000
7144,Юнусабадский,2,61,2,5,42000
...,...,...,...,...,...,...
2402,Шайхантахурский,2,46,5,9,24500
4316,Мирабадский,2,69,9,10,62000
5628,Мирабадский,4,90,2,4,105000
3283,Олмазорский,4,140,5,5,110000


In [159]:
X_test = test_set.drop('price', axis=1)
y_test = test_set['price'].copy()

In [160]:
test_set

,district,rooms,size,level,max_levels,price
6214,Чиланзарский,2,60,4,9,34500
3311,Мирзо-Улугбекский,2,50,1,4,29500
3066,Мирзо-Улугбекский,3,152,5,8,285000
1896,Юнусабадский,3,78,4,4,55000
7144,Юнусабадский,2,61,2,5,42000
...,...,...,...,...,...,...
2402,Шайхантахурский,2,46,5,9,24500
4316,Мирабадский,2,69,9,10,62000
5628,Мирабадский,4,90,2,4,105000
3283,Олмазорский,4,140,5,5,110000


In [161]:
X_test_prepered = full_pipeline.transform(X_test)

In [162]:
y_predict = LR_model.predict(X_test_prepered)

In [163]:
from sklearn.metrics import mean_absolute_error

mea = mean_absolute_error(y_test, y_predict)
print(f"o'rtacha absolut hatolik {mea}")

o'rtacha absolut hatolik 61642.58189451634


In [144]:
y

6038    45000
2058    74000
7114    48500
5035    57500
6864    27000
        ...  
500     74000
3885    30000
7085    78000
4517    55000
5925    23000
Name: price, Length: 6052, dtype: int64

In [145]:
LR_model.score(X_test_prepered,y_test)*100

0.030032024422166348